# Paper 1 — Golden Age Semantic Reconfiguration

**Project:** *Reconfiguring the Golden Age: Semantic Networks and the Renaissance–Baroque Transition in Spanish Poetry*

This is the **single working notebook** for Paper 1. The GitHub repository is the source of truth. We will keep the notebook as the analysis dashboard and progressively move reusable scientific logic into `src/`.


## Colab ↔ GitHub workflow

1. Open this notebook from the GitHub repository in Google Colab.
2. Run the analysis here.
3. Preserve a meaningful state with **File → Save a copy in GitHub**, overwriting `notebooks/paper1_analysis.ipynb` on `main`.
4. Do **not** create parallel experimental notebooks. Parameter changes will later be handled through configuration files.

> Colab does not continuously auto-save to GitHub. The deliberate GitHub save is useful because each preserved state becomes versionable and reproducible.


## 00. Environment & reproducibility

We first record the runtime and import only the packages needed for the corpus audit.


In [3]:
import sys
import os
import re
import json
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')
print(f'Python: {sys.version.split()[0]}')
print(f'pandas: {pd.__version__}')


Running in Colab: True
Python: 3.13.15
pandas: 2.2.3


## 01. Sprint 1 — Corpus Audit + Temporal Reconstruction

### Scientific objective

Before constructing any semantic network, we need a poem-level master table with, ideally,

`poem_id, author, title, text, source, date_min, date_max, publication_year, temporal_confidence, period`.

The first scientific result of the project is therefore **not a network**. It is a defensible audit of the corpus and its temporal metadata.


### 01.1 Authoritative corpus sources

We will initially audit **two related but distinct sources**:

1. **Hernández-Lorenzo network corpus** — the exact corpus used in the prior network study, including Herrera and Pacheco and a metadata file.
2. **CorpusSonetosSigloDeOro (Navarro Colorado)** — poem-level TEI/XML files, useful for reconstructing individual sonnets and source metadata.

Both repositories are pinned to specific commits. This prevents future upstream changes from silently altering our results.


In [4]:
SOURCES = {
    "hernandez_network": {
        "repo": "https://github.com/lamusadecima/Network_for_Golden_Age_Spanish_Poetry.git",
        "commit": "ef6b7b691f67abe60d9cfa85c274f0be8095dd9a",
    },
    "navarro_tei": {
        "repo": "https://github.com/bncolorado/CorpusSonetosSigloDeOro.git",
        "commit": "092a5fe70a4065a4d84bfed288bffd3851348f9c",
    },
}

SOURCE_ROOT = Path("/content/gasr_sources")
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)

SOURCES


{'hernandez_network': {'repo': 'https://github.com/lamusadecima/Network_for_Golden_Age_Spanish_Poetry.git',
  'commit': 'ef6b7b691f67abe60d9cfa85c274f0be8095dd9a'},
 'navarro_tei': {'repo': 'https://github.com/bncolorado/CorpusSonetosSigloDeOro.git',
  'commit': '092a5fe70a4065a4d84bfed288bffd3851348f9c'}}

In [5]:
def clone_at_commit(name, repo_url, commit):
    target = SOURCE_ROOT / name

    if target.exists():
        shutil.rmtree(target)

    subprocess.run(
        ["git", "clone", "--quiet", repo_url, str(target)],
        check=True
    )
    subprocess.run(
        ["git", "-C", str(target), "checkout", "--quiet", commit],
        check=True
    )

    resolved = subprocess.check_output(
        ["git", "-C", str(target), "rev-parse", "HEAD"],
        text=True
    ).strip()

    assert resolved == commit, (name, resolved, commit)
    return target

source_paths = {
    name: clone_at_commit(name, info["repo"], info["commit"])
    for name, info in SOURCES.items()
}

print("Pinned sources ready:")
for name, path in source_paths.items():
    print(f"  {name}: {path}")


Pinned sources ready:
  hernandez_network: /content/gasr_sources/hernandez_network
  navarro_tei: /content/gasr_sources/navarro_tei


If the previous cell ends with `Pinned sources ready`, we are analysing exactly the intended upstream versions. No GitHub token is required here because both source corpora are public.


### 01.2 File-level census

We now ask a very basic but essential question: **what is actually present in each source?**


In [6]:
def file_inventory(root: Path):
    rows = []
    for p in root.rglob("*"):
        if p.is_file() and ".git" not in p.parts:
            rows.append({
                "path": str(p.relative_to(root)),
                "suffix": p.suffix.lower(),
                "bytes": p.stat().st_size,
                "parent": p.parent.name,
            })
    return pd.DataFrame(rows)

inventories = {
    name: file_inventory(path)
    for name, path in source_paths.items()
}

for name, inv in inventories.items():
    print(f"\n{name}")
    print("-" * len(name))
    print(f"Files: {len(inv):,}")
    print(inv["suffix"].value_counts(dropna=False).to_string())



hernandez_network
-----------------
Files: 45
suffix
.txt    40
.csv     2
.md      1
.r       1
.png     1

navarro_tei
-----------
Files: 5,081
suffix
.xml    5078
.md        2
.pdf       1


### 01.3 Audit of the Hernández-Lorenzo corpus

The prior study aggregates sonnets by author in plain-text files. We first enumerate these files and search the repository for likely metadata tables.


In [7]:
hernandez_root = source_paths["hernandez_network"]
hernandez_corpus_dir = hernandez_root / "corpus"

txt_files = sorted(hernandez_corpus_dir.glob("*.txt"))
print(f"TXT files in corpus/: {len(txt_files)}")

txt_audit = pd.DataFrame([
    {
        "file": p.name,
        "author_raw": re.sub(r"_?Sonetos.*$", "", p.stem, flags=re.I),
        "bytes": p.stat().st_size,
        "lines": len(p.read_text(encoding="utf-8", errors="replace").splitlines()),
        "characters": len(p.read_text(encoding="utf-8", errors="replace")),
    }
    for p in txt_files
]).sort_values("author_raw")

display(txt_audit.head(15))
print(f"\nTotal characters across aggregated TXT corpus: {txt_audit['characters'].sum():,}")


TXT files in corpus/: 40


,file,author_raw,bytes,lines,characters
0,AN_SonetosP2.txt,AN,112388,2729,108248
1,AgustinSalazar_Sonetos.txt,AgustinSalazar,17701,450,16941
2,AntonioEnriquezGomez_Sonetos.txt,AntonioEnriquezGomez,22515,602,22175
3,BartolomeArgensola_Sonetos.txt,BartolomeArgensola,92621,2369,91194
4,BernardinodeRebolledo_Sonetos.txt,BernardinodeRebolledo,29926,809,29539
5,BocangelyUnzueta_Sonetos.txt,BocangelyUnzueta,44833,1154,44085
6,Cervantes_Sonetos.txt,Cervantes,45338,1162,44644
7,DiegoHurtadodeMendoza_Sonetos.txt,DiegoHurtadodeMendoza,34382,901,33811
8,DiegoXimenezAyllon_Sonetos.txt,DiegoXimenezAyllon,31242,824,30856
9,FranciscoDelaTorre_Sonetos.txt,FranciscoDelaTorre,44275,1170,43741



Total characters across aggregated TXT corpus: 2,500,531


In [8]:
# Search every non-git file for filenames that may encode metadata.
keywords = re.compile(r"(meta|author|autor|info|date|fecha|birth|nacimiento|node|table)", re.I)

metadata_candidates = []
for p in hernandez_root.rglob("*"):
    if p.is_file() and ".git" not in p.parts and keywords.search(p.name):
        metadata_candidates.append({
            "path": str(p.relative_to(hernandez_root)),
            "suffix": p.suffix.lower(),
            "bytes": p.stat().st_size,
        })

metadata_candidates = pd.DataFrame(metadata_candidates)
display(metadata_candidates if len(metadata_candidates) else pd.DataFrame(
    {"message": ["No filename-based metadata candidate found; inspect full tree below."]}
))

print("\nTop-level and corpus files:")
for p in sorted(hernandez_root.rglob("*")):
    if p.is_file() and ".git" not in p.parts:
        rel = p.relative_to(hernandez_root)
        if len(rel.parts) <= 2:
            print(rel)


,path,suffix,bytes
0,corpus/BernardinodeRebolledo_Sonetos.txt,.txt,29926
1,corpus/metadata.csv,.csv,1872
2,materials/ADSO table3.csv,.csv,7609



Top-level and corpus files:
README.md
code/Script Networks.R
corpus/AN_SonetosP2.txt
corpus/AgustinSalazar_Sonetos.txt
corpus/AntonioEnriquezGomez_Sonetos.txt
corpus/BartolomeArgensola_Sonetos.txt
corpus/BernardinodeRebolledo_Sonetos.txt
corpus/BocangelyUnzueta_Sonetos.txt
corpus/Cervantes_Sonetos.txt
corpus/DiegoHurtadodeMendoza_Sonetos.txt
corpus/DiegoXimenezAyllon_Sonetos.txt
corpus/FranciscoDelaTorre_Sonetos.txt
corpus/FranciscodeAldana_Sonetos.txt
corpus/FranciscodeBorja_Sonetos.txt
corpus/FranciscodeMedrano_Sonetos.txt
corpus/Garcilaso_Sonetos.txt
corpus/Gongora_Sonetos.txt
corpus/GutierredeCetina_Sonetos.txt
corpus/HernandodeAcunya_Sonetos.txt
corpus/Herrera_Sonetos.txt
corpus/JosephdeLitalayCastelvi_Sonetos.txt
corpus/JuanBoscan_Sonetos.txt
corpus/JuandeAlmeida_Sonetos.txt
corpus/JuandeArguijo_Sonetos.txt
corpus/JuandeJauregui_Sonetos.txt
corpus/JuandeTassisyPeralta_Sonetos.txt
corpus/JuandeTimoneda_Sonetos.txt
corpus/LopedeVega_Sonetos.txt
corpus/LopedeZarate_Sonetos.txt
corp

### 01.4 Inspect one aggregated text file

This tells us whether poem boundaries/titles survive in the plain-text derivative or whether we must reconstruct poem-level observations primarily from TEI/XML.


In [9]:
preferred = hernandez_corpus_dir / "Cervantes_Sonetos.txt"
sample_txt = preferred if preferred.exists() else txt_files[0]

print(f"Sample: {sample_txt.name}\n")
sample_lines = sample_txt.read_text(encoding="utf-8", errors="replace").splitlines()

for i, line in enumerate(sample_lines[:80], start=1):
    print(f"{i:03d}: {line}")


Sample: Cervantes_Sonetos.txt

001:     Tú, que imitaste la llorosa vida
002:     que tuve, ausente y desdeñado, sobre
003:     el gran ribazo de la Peña Pobre.
004:     De alegre a penitencia reducida;
005:     tú, a quien los ojos dieron la bebida
006:     de abundante licor, aunque salobre;
007:     y alzándote la plata, estaño y cobre,
008:     te dio la tierra en tierra la comida;
009:     vive seguro de que eternamente
010:     (en tanto al menos que en la cuarta esfera,
011:     sus cabellos aguije el bello Apolo)
012:     Tendrás claro renombre de valiente;
013:     tu patria será en todas la primera,
014:     tu sabio autor al mundo único y solo.
015:     
016:     El muro rompe la doncella hermosa
017:     que de Píramo abrió el gallardo pecho;
018:     parte el amor de Chipre, y va derecho
019:     a ver la quiebra estrecha y prodigiosa.
020:     Habla el silencio allí, porque no osa
021:     la voz entrar por tan estrecho estrecho.
022:     Las almas sí, que amor suele de h

### 01.5 Poem-level census of CorpusSonetosSigloDeOro

The Navarro corpus stores sonnets as individual XML files under author directories. This is likely the most useful starting point for our **poem-level** master table.


In [10]:
navarro_root = source_paths["navarro_tei"]
xml_files = sorted([
    p for p in navarro_root.rglob("*.xml")
    if ".git" not in p.parts
])

tei_census = pd.DataFrame([
    {
        "author_dir": p.parent.name,
        "file": p.name,
        "path": str(p.relative_to(navarro_root)),
        "bytes": p.stat().st_size,
    }
    for p in xml_files
])

print(f"XML sonnet files: {len(tei_census):,}")
print(f"Author directories: {tei_census['author_dir'].nunique():,}")
display(
    tei_census.groupby("author_dir")
    .size()
    .rename("n_sonnets")
    .sort_values(ascending=False)
    .reset_index()
    .head(20)
)


XML sonnet files: 5,078
Author directories: 53


,author_dir,n_sonnets
0,LopeDeVega_1,699
1,LopeDeVega_2,647
2,Quevedo,517
3,FernandoDeHerrera,320
4,GutierreDeCetina,248
5,JuanDeTassisYPeralta,203
6,ArgensolaBartolome,158
7,JosephDeLitalaYCastelvi,152
8,FcodeBorja,142
9,PedroSotoDeRojas,125


### 01.6 Parse TEI into a provisional poem-level table

At this stage we extract only information explicitly present in the TEI: author folder, title, verse text, bibliographic source, and line count. We **do not invent dates**.


In [11]:
TEI_NS = {"tei": "http://www.tei-c.org/ns/1.0"}

def get_text(el):
    if el is None:
        return None
    return " ".join("".join(el.itertext()).split())

def parse_tei_poem(path: Path, root: Path):
    tree = ET.parse(path)
    tei = tree.getroot()

    title_el = tei.find(".//tei:text/tei:body/tei:head/tei:title", TEI_NS)
    bibl_el = tei.find(".//tei:sourceDesc/tei:bibl", TEI_NS)
    author_el = tei.find(".//tei:sourceDesc//tei:author", TEI_NS)
    lines = tei.findall(".//tei:text/tei:body//tei:l", TEI_NS)

    verses = [get_text(x) for x in lines]
    verses = [x for x in verses if x]

    return {
        "poem_id": str(path.relative_to(root)).replace("/", "__").replace(".xml", ""),
        "author_dir": path.parent.name,
        "author_tei": get_text(author_el),
        "title": get_text(title_el),
        "text": "\n".join(verses),
        "n_lines": len(verses),
        "source_bibl": get_text(bibl_el),
        "source_file": str(path.relative_to(root)),
        "date_min": pd.NA,
        "date_max": pd.NA,
        "publication_year": pd.NA,
        "temporal_confidence": "unassigned",
    }

records = []
parse_errors = []

for p in xml_files:
    try:
        records.append(parse_tei_poem(p, navarro_root))
    except Exception as exc:
        parse_errors.append((str(p.relative_to(navarro_root)), repr(exc)))

poems = pd.DataFrame(records)

print(f"Parsed poems: {len(poems):,}")
print(f"Parse errors: {len(parse_errors):,}")
print(f"Authors/folders: {poems['author_dir'].nunique():,}")
print(f"14-line sonnets: {(poems['n_lines'] == 14).sum():,} / {len(poems):,}")

display(poems[["poem_id", "author_dir", "author_tei", "title", "n_lines", "source_bibl"]].head(10))


Parsed poems: 5,078
Parse errors: 0
Authors/folders: 53
14-line sonnets: 5,053 / 5,078


,poem_id,author_dir,author_tei,title,n_lines,source_bibl
0,AgustinDeSalazarYTorres__AgustinDeSalazarYTorr...,AgustinDeSalazarYTorres,Agustin de Salazar y Torres,- I - Da noticias de sus gracias,14,Sonetos de Agustin de Salazar y Torres. Biblio...
1,AgustinDeSalazarYTorres__AgustinDeSalazarYTorr...,AgustinDeSalazarYTorres,Agustin de Salazar y Torres,"- XVIII - A Cintia, que mirando unos lienzos, ...",14,Sonetos de Agustin de Salazar y Torres. Biblio...
2,AgustinDeSalazarYTorres__AgustinDeSalazarYTorr...,AgustinDeSalazarYTorres,Agustin de Salazar y Torres,- XIX - Amor compadecido del dolor de un amant...,14,Sonetos de Agustin de Salazar y Torres. Biblio...
3,AgustinDeSalazarYTorres__AgustinDeSalazarYTorr...,AgustinDeSalazarYTorres,Agustin de Salazar y Torres,- II - Muestras de amor,14,Sonetos de Agustin de Salazar y Torres. Biblio...
4,AgustinDeSalazarYTorres__AgustinDeSalazarYTorr...,AgustinDeSalazarYTorres,Agustin de Salazar y Torres,- XX - Al poder mas que soberano de los ojos d...,14,Sonetos de Agustin de Salazar y Torres. Biblio...
5,AgustinDeSalazarYTorres__AgustinDeSalazarYTorr...,AgustinDeSalazarYTorres,Agustin de Salazar y Torres,"- XXI - Un amante, soñando que su dama era mue...",14,Sonetos de Agustin de Salazar y Torres. Biblio...
6,AgustinDeSalazarYTorres__AgustinDeSalazarYTorr...,AgustinDeSalazarYTorres,Agustin de Salazar y Torres,- XXII -,14,Sonetos de Agustin de Salazar y Torres. Biblio...
7,AgustinDeSalazarYTorres__AgustinDeSalazarYTorr...,AgustinDeSalazarYTorres,Agustin de Salazar y Torres,"- XXIII - A un retrato de Cintia, antes de hab...",14,Sonetos de Agustin de Salazar y Torres. Biblio...
8,AgustinDeSalazarYTorres__AgustinDeSalazarYTorr...,AgustinDeSalazarYTorres,Agustin de Salazar y Torres,- XXIV - A una dama de pie demasiado crecido,14,Sonetos de Agustin de Salazar y Torres. Biblio...
9,AgustinDeSalazarYTorres__AgustinDeSalazarYTorr...,AgustinDeSalazarYTorres,Agustin de Salazar y Torres,"- XXV - A una dama que, siendo vieja, tenía ma...",14,Sonetos de Agustin de Salazar y Torres. Biblio...


### 01.7 Temporal metadata audit

This is the decisive checkpoint. We search the TEI files for explicit date-like tags/attributes and inspect the bibliographic descriptions. If dates are absent, chronology must be reconstructed from external scholarly metadata rather than inferred from author birth year.


In [12]:
DATE_TAG_PATTERNS = re.compile(r"(date|when|notBefore|notAfter|from|to)", re.I)

date_hits = []
for p in xml_files:
    raw = p.read_text(encoding="utf-8", errors="replace")
    if DATE_TAG_PATTERNS.search(raw):
        snippets = []
        for line in raw.splitlines():
            if DATE_TAG_PATTERNS.search(line):
                snippets.append(line.strip())
        date_hits.append({
            "file": str(p.relative_to(navarro_root)),
            "hits": " | ".join(snippets[:5])
        })

date_hits_df = pd.DataFrame(date_hits)

print(f"XML files containing date-like strings/tags: {len(date_hits_df):,}")
display(date_hits_df.head(20) if len(date_hits_df) else pd.DataFrame(
    {"result": ["No explicit date-like TEI fields detected."]}
))


XML files containing date-like strings/tags: 5,078


,file,hits
0,AgustinDeSalazarYTorres/AgustinDeSalazarYTorre...,<bibl><title>Sonetos</title> de <author>Agusti...
1,AgustinDeSalazarYTorres/AgustinDeSalazarYTorre...,<bibl><title>Sonetos</title> de <author>Agusti...
2,AgustinDeSalazarYTorres/AgustinDeSalazarYTorre...,<bibl><title>Sonetos</title> de <author>Agusti...
3,AgustinDeSalazarYTorres/AgustinDeSalazarYTorre...,<bibl><title>Sonetos</title> de <author>Agusti...
4,AgustinDeSalazarYTorres/AgustinDeSalazarYTorre...,<bibl><title>Sonetos</title> de <author>Agusti...
5,AgustinDeSalazarYTorres/AgustinDeSalazarYTorre...,<bibl><title>Sonetos</title> de <author>Agusti...
6,AgustinDeSalazarYTorres/AgustinDeSalazarYTorre...,<bibl><title>Sonetos</title> de <author>Agusti...
7,AgustinDeSalazarYTorres/AgustinDeSalazarYTorre...,<bibl><title>Sonetos</title> de <author>Agusti...
8,AgustinDeSalazarYTorres/AgustinDeSalazarYTorre...,<bibl><title>Sonetos</title> de <author>Agusti...
9,AgustinDeSalazarYTorres/AgustinDeSalazarYTorre...,<bibl><title>Sonetos</title> de <author>Agusti...


### 01.8 Corpus audit summary

Run this cell after all previous cells. It creates a compact checkpoint that we will use to decide the temporal-reconstruction strategy.


In [13]:
audit_summary = pd.DataFrame([
    {
        "source": "Hernández-Lorenzo network corpus",
        "unit_detected": "author-level TXT",
        "n_files": len(txt_files),
        "n_authors_or_dirs": txt_audit["author_raw"].nunique(),
        "role_in_paper1": "prior-study replication/reference + Herrera/Pacheco additions",
    },
    {
        "source": "Navarro CorpusSonetosSigloDeOro",
        "unit_detected": "poem-level TEI/XML",
        "n_files": len(xml_files),
        "n_authors_or_dirs": poems["author_dir"].nunique(),
        "role_in_paper1": "poem-level reconstruction + textual/bibliographic backbone",
    },
])

display(audit_summary)

print("\nTemporal fields currently populated in provisional poem table:")
for col in ["date_min", "date_max", "publication_year", "temporal_confidence"]:
    print(f"  {col}: {poems[col].notna().sum():,} non-null")

print("\nCHECKPOINT:")
print("Do not build semantic networks yet.")
print("Save this executed notebook to GitHub and inspect the outputs above.")


,source,unit_detected,n_files,n_authors_or_dirs,role_in_paper1
0,Hernández-Lorenzo network corpus,author-level TXT,40,40,prior-study replication/reference + Herrera/Pa...
1,Navarro CorpusSonetosSigloDeOro,poem-level TEI/XML,5078,53,poem-level reconstruction + textual/bibliograp...



Temporal fields currently populated in provisional poem table:
  date_min: 0 non-null
  date_max: 0 non-null
  publication_year: 0 non-null
  temporal_confidence: 5,078 non-null

CHECKPOINT:
Do not build semantic networks yet.
Save this executed notebook to GitHub and inspect the outputs above.


---

## What we will decide from this execution

After saving this run to GitHub, the next step will be based on the **observed outputs**, not assumptions:

- whether the Hernández metadata file can contribute dates;
- whether poem boundaries survive in the aggregated TXT corpus;
- how many poem-level TEI records and authors are available;
- whether explicit dating exists in TEI;
- which authors in Hernández are absent from Navarro (especially Herrera/Pacheco);
- and therefore what temporal-reconstruction strategy is defensible.

Only after that audit will we implement the master chronology.


## Planned later sections

- 02 — Corpus reconciliation
- 03 — Temporal metadata reconstruction
- 04 — Linguistic preprocessing
- 05 — Semantic network construction
- 06 — Temporal semantic networks
- 07 — Structural reconfiguration
- 08 — Change-point detection
- 09 — Author ablation
- 10 — Concept trajectories
- 11 — Robustness checks
- 12 — Publication figures
- 13 — Export tables
